# This notebook provides a minimal example on how to use SysVar.
# We will create Eigenvariations of our fitting variable based on the charged slow pion corrections in 2 reconstruction channels and for 2 templates in 15 bins

In [1]:
%reload_ext autoreload
%autoreload 2

# Attention: 

### Setting the SysVar Path

**Please update the path below to point to the location where you have installed your SysVar fork, then execute the cell.**

This step is currently necessary because the basf2 developers have renamed their `pidvar` class to `sysvar`, anticipating an early merge of the SysVar package into basf2.

If you are running with an environment sourced from CVMFS and have not executed the cell below, you will encounter the following error:

```
ModuleNotFoundError: No module named 'sysvar.utils'; 'sysvar' is not a package
```

**To avoid this error, set the correct path to your SysVar installation and run the cell.**

In [2]:
import sys
sys.path.insert(0,'{path_where_you_pip_installed_sysvar}/SysVar/src')

# Create a pseudo-dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

sample_size = 2000
theta =  5.5
momentum_scaler =MinMaxScaler((0.05, 0.4))
momentum_scaler.fit(np.random.gamma(theta, 1.0, sample_size).reshape(-1, 1))
toy_df = pd.DataFrame(
    {
        "channel": np.random.randint(0, 2, sample_size), # Assume 4 reco channels,
        "template": np.random.randint(1, 3, sample_size), # Assume 2 templates, 1 = signal and 2 = BKG
        "slow_pi_p": momentum_scaler.transform(np.random.gamma(theta, 1.0, sample_size).reshape(-1, 1)).flatten(),
        "slow_pi2_p": momentum_scaler.transform(np.random.gamma(theta, 1.0, sample_size).reshape(-1, 1)).flatten()
    }
)

toy_df["slow_pi_mcPDG"] =  np.random.choice([-211, 211], sample_size)
toy_df["slow_pi2_mcPDG"] =  np.random.choice([-211, 211], sample_size)
toy_df["slow_pi_PDG"] =  np.random.choice([-211, 211], sample_size)
toy_df["slow_pi2_PDG"] =  np.random.choice([-211, 211], sample_size)

toy_df.loc[toy_df.template == 1, "fit_variable1"] = np.random.exponential(0.2, len(toy_df[toy_df.template == 1]))
toy_df.loc[toy_df.template == 1, "fit_variable2"] = np.random.normal(2.5, 0.3, len(toy_df[toy_df.template == 1]))
toy_df.loc[toy_df.template == 1, "other_weight"] = np.random.normal(0.3, 0.04, len(toy_df[toy_df.template == 1]))
toy_df.loc[toy_df.template == 2, "fit_variable1"] = np.random.power(1.5, len(toy_df[toy_df.template == 2]))
toy_df.loc[toy_df.template == 2, "fit_variable2"] = np.random.rayleigh(1.5, size = len(toy_df[toy_df.template == 2]))
toy_df.loc[toy_df.template == 2, "other_weight"] = np.random.normal(0.8, 0.1, len(toy_df[toy_df.template == 2]))

toy_df = toy_df.query("0 < fit_variable1 < 1")
toy_df = toy_df.query("1 < fit_variable2 < 4")
toy_df = toy_df.query("0.05 < slow_pi_p < 0.4")
toy_df = toy_df.query("0.05 < slow_pi2_p < 0.4")

toy_df["template"].replace(1, "signal", inplace = True)
toy_df["template"].replace(2, "bkg", inplace = True)

toy_df


# Now let's add slow pion weights to the dataframe

In [ ]:
from sysvar import add_weights_to_dataframe

add_weights_to_dataframe(
    df = toy_df,
    systematic= "charged_slow_pi",
    MC_production= "MC15rd",
    prefix= "slow_pi",
    weightname ="charged_weight",
    #overwrite: False,
    #Nvar: 0
)
toy_df

If you try again, then SysVar will complain. In order to overwrite an existing weight, set the relevant argument to True

In [ ]:
add_weights_to_dataframe(
    df = toy_df,
    systematic= "charged_slow_pi",
    MC_production= "MC15rd",
    prefix= "slow_pi",
    weightname ="charged_weight",
    #overwrite: False
)

Now we create the total weight of the analysis

In [6]:
toy_df["total_weight"] = toy_df[["other_weight", "slow_pi_charged_weight"]].product(axis = 1)

# Let's plot what we have

In [ ]:
PALETTE = sns.color_palette("colorblind")


def plot_var_in_2_channels(column_name, xlabel, bins, scope):

    fig, ax = plt.subplots(1, 2, figsize = (16, 4.5), dpi = 200)

    for i, c in enumerate(toy_df.channel.value_counts().keys()):
        for j, t in enumerate(toy_df.template.value_counts().keys()):
            tmp_df = toy_df.query(f"channel == {c} & template == '{t}'")
            ax[i].hist(
                tmp_df[column_name],
                weights=tmp_df["total_weight"],
                histtype = "step",
                label = "Signal" if t == "signal" else "BKG",
                bins = bins,
                range = scope,
                linewidth = 2,
                color = PALETTE[j]
            )
        ax[i].legend()
        ax[i].set_xlabel(xlabel)
        ax[i].set_ylabel("Events")
        ax[i].set_title(f"Reco channel {i+1}")


plot_var_in_2_channels("fit_variable1", "Pseudo fit-variable 1", bins = 5, scope = (0,1))
plt.tight_layout()
#plt.savefig("pseudofit1.png", dpi = 200)
plot_var_in_2_channels("fit_variable2", "Pseudo fit-variable 2", bins = 5, scope = (1,4))
plt.tight_layout()
#plt.savefig("pseudofit2.png", dpi = 200)
plot_var_in_2_channels("slow_pi_p", r"$p_{\pi_{s}}$ in GeV", bins = [0.05, 0.12, 0.16, 0.2, 0.4], scope = (0, 0.4))
plt.tight_layout()

# Define the configurations for the Eigen decomposition


### Instructions for the SysVar Configuration File

This configuration file is the primary interface for communicating with SysVar. Your entire analysis setup should be encoded here.

#### Parameters

- **`output_filepath`**:  
  The path where both nominal templates and all template variations will be saved.

- **`reco_channel_id_column`**:  
  The name of the column in your dataframe that distinguishes different reconstruction channels to be fitted separately but simultaneously.

- **`reco_channels`**:  
  A dictionary mapping custom channel names (as keys) to lists of values to be found in the `reco_channel_id_column`.

- **`template_id_column`**:  
  The name of the column in your dataframe that distinguishes different templates.

- **`templates`**:  
  A list of template names. These should correspond to values found in the `template_id_column` of your dataframe.

- **`total_weight`**:  
  The total weight to use when creating nominal templates for your analysis.

- **`MC_prod`**:  
  The Belle II MC campaign. Currently, only `MC15rd` is supported.

- **`Nvar`**:  
  The number of variation weights to generate.

- **`bins`**:  
  A dictionary (of dictionaries) defining the binning for each channel and variable. Both 1D and ND histogram projections are supported.

- **`systematics`**:  
  A dictionary where each key is the name of a systematic.  
    - There should be a corresponding YAML file in the configs (see [configs/MC15rd](https://gitlab.desy.de/itsaklid/sysvar/-/tree/main/configs/MC15rd?ref_type=heads)).
    - The key `weight` should contain the name of the column with the weight associated with this systematic.
    - `prefices`: A string or list of strings to prepend to the weight name to build the column name present in the dataframe.
    - `reco_channels`: Specify a list of channels (as strings) to include or exclude in the systematic variation. If both are `None`, all channels will be affected.
    - `templates`: List of affected templates for this systematic. Defaults to `None` (all templates affected).

---

Please find below the configuration file that corresponds to this mininal example

In [ ]:
from sysvar.utils import read_yaml

settings = read_yaml("study_setup", "MC15rd")
settings

# First let's save the nominal templates. 
This should always be the first step! It should always be prefered that the nominal histograms are saved before any variations in order to avoid problems with uproot's behavior when saving and updating files

In [ ]:
from sysvar import save_nominal_templates
save_nominal_templates(toy_df, settings)

# Let's now perform the eigendecomposition. 
The eigendecompose helper function will return an object which is the only input needed to use the rest of the API

In [ ]:
from sysvar import eigendecompose

egd = eigendecompose(
    df = toy_df,
    settings = settings,
    syst_effect = "charged_slow_pi",
    #criterion: "max_differences",
    #prec: 0.005,
    #save_variations: False
)

# Examine the correlation matrix
The dimensions here are (N_channels * N_templates * Nbins). 

In this case 2x2x15 = 60

In [ ]:
from sysvar import plot_analysis_corr_matrix
plot_analysis_corr_matrix(egd)

# Visualize how many of eigenvariations are important
We will also try to save the figure. This will fail, but have a look at the end of the error message

In [ ]:
from sysvar import plot_cov_diff
plot_cov_diff(egd, save=True, filename = "test_figure")

In [ ]:
from sysvar import register_saving_info

si = {
    "top_dir": "./"
}
register_saving_info(egd, si)

# Now let's try again
Notice that the saving is not failing now, but we are informed that we can more control over the way that the figure are being saved

In [ ]:
plot_cov_diff(egd, save=True, filename = "test_figure")

# Save the templates in the format cabinetry expects
If you did not enable save_variations = True because you were not sure that you want to save the variation you can still do this since you have access to the eigenvariation object.

In [ ]:
# Now save the templates variations throught the EigenDecomposer object
egd.save_template_variations()

Let's examine a bit more closely the templates that we have created

In [ ]:
from sysvar import plot_templates_relative_variations_in_grid, plot_up_and_down_variations
plot_up_and_down_variations(egd)
plot_templates_relative_variations_in_grid(egd)

# Now how did we get those variations ? 

# Every EigenDecomposer object creates a Variator object which implements a covariance matrix between the different correction bins

This ensures that the variations that we sample from are common for every template

In [ ]:
from sysvar import plot_correction_variations_in_grid, plot_correction_cov_and_corr
plot_correction_variations_in_grid(egd)
plot_correction_cov_and_corr(egd)

# What about the corrections itself ? 

In [ ]:
from sysvar import plot_correction_errors
plot_correction_errors(egd)